## Result Visualisation 

In [ ]:
import os
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

import torch
import torch.nn as nn
from torch.optim import AdamW, SGD

from vision_studio.dataset import ImageNetClassificationDataset
from vision_studio.data_loader import SimpleDataLoader
from vision_studio.models import ResNet50, SwinTransformer, CustomModel
from vision_studio.models.base import BaseModel
from vision_studio.evaluate import ClassificationEvaluator
from vision_studio.augmentation import Compose, Resize
from vision_studio.types import SwinModelSelect

from torchvision.models import ResNet50_Weights, Swin_V2_T_Weights

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 10

In [ ]:
# Setup dataset paths and validation data
dataset_path = "/mnt/d/Traffic_Sign_Dataset"  # Adjust path if needed
model_checkpoint_dir = "/home/mozi30/data/models"

# Create resize transform
resize = Compose([
    Resize(size=(224, 224))
])

# Load validation dataset
dataset_val = ImageNetClassificationDataset(
    root_dir=dataset_path,
    split="val",
    transform=resize
)

# Create validation dataloader
val_loader = SimpleDataLoader(
    dataset=dataset_val,
    batch_size=64,
    shuffle=False
)

num_classes = dataset_val.get_num_classes()
print(f"Validation dataset size: {len(dataset_val)}")
print(f"Number of classes: {num_classes}")
print(f"Class names: {dataset_val.class_names[:5]}... (showing first 5)")

In [ ]:
# Model builders and validation function
def build_resnet50() -> ResNet50:
    return ResNet50(num_classes=num_classes, weights=ResNet50_Weights.IMAGENET1K_V2)


def build_swin_v2_t() -> SwinTransformer:
    return SwinTransformer(
        model_select=SwinModelSelect.SWIN_V2_T,
        num_classes=num_classes,
        weights=Swin_V2_T_Weights.IMAGENET1K_V1
    )


def build_custom_model() -> CustomModel:
    return CustomModel(input_shape=(3, 224, 224), num_classes=num_classes)


def validate_model(model: BaseModel, val_loader, device: str = "cuda") -> dict[str, Any]:
    """Validate a model on validation set and return metrics."""
    model.to(device)
    model.eval()
    
    evaluator = ClassificationEvaluator(num_classes=num_classes, topk=(1, 5))
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(val_loader):
            inputs, targets = batch
            inputs = inputs.to(device)
            targets_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                            for k, v in targets.items()}
            
            logits = model(inputs)
            preds = logits.detach().cpu()
            labels = targets["label"].cpu()
            loss = torch.tensor(0.0)  # Dummy loss for evaluator
            
            evaluator.update(preds, labels, loss)
            
            if (batch_idx + 1) % 10 == 0:
                print(f"  Validated batch {batch_idx + 1}/{len(val_loader)}")
    
    metrics = evaluator.compute()
    metrics["confusion_matrix"] = evaluator.confusion_matrix.numpy()
    
    return metrics


# Define models to validate
models_to_validate = [
    {
        "name": "ResNet50 (base)",
        "builder": build_resnet50,
        "checkpoint": None  # Will use pretrained weights
    },
    {
        "name": "SwinTransformer V2-T (base)",
        "builder": build_swin_v2_t,
        "checkpoint": None
    },
    {
        "name": "CustomModel (base)",
        "builder": build_custom_model,
        "checkpoint": f"{model_checkpoint_dir}/myModel-base.pth"
    },
    {
        "name": "CustomModel (finetuned)",
        "builder": build_custom_model,
        "checkpoint": f"{model_checkpoint_dir}/myModel-finetune-base.pth"
    },
]

In [ ]:
# Validate all models and collect metrics
validation_results = {}

for model_config in models_to_validate:
    model_name = model_config["name"]
    print(f"\nValidating {model_name}...")
    
    # Build model
    model = model_config["builder"]()
    
    # Load checkpoint if available
    if model_config["checkpoint"] and os.path.exists(model_config["checkpoint"]):
        print(f"  Loading checkpoint: {model_config['checkpoint']}")
        model.load_checkpoint(model_config["checkpoint"])
    
    # Validate
    metrics = validate_model(model, val_loader, device="cuda")
    validation_results[model_name] = metrics
    
    print(f"  ✓ Accuracy: {metrics['accuracy']:.4f}")
    print(f"  ✓ F1-Score (macro): {metrics['f1_macro']:.4f}")
    print(f"  ✓ Precision (macro): {metrics['precision_macro']:.4f}")
    print(f"  ✓ Recall (macro): {metrics['recall_macro']:.4f}")

print("\n" + "="*60)
print("Validation complete!")
print("="*60)

In [ ]:
# Create comparison dataframe
comparison_data = []
for model_name, metrics in validation_results.items():
    comparison_data.append({
        "Model": model_name,
        "Accuracy": metrics["accuracy"],
        "F1-Score (Macro)": metrics["f1_macro"],
        "F1-Score (Micro)": metrics["f1_micro"],
        "Precision (Macro)": metrics["precision_macro"],
        "Recall (Macro)": metrics["recall_macro"],
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("Model Comparison Results")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

In [ ]:
# Plot model comparison - Accuracy and F1-Scores
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Model Performance Comparison", fontsize=16, fontweight="bold")

# Accuracy comparison
ax = axes[0, 0]
models = comparison_df["Model"].values
accuracy = comparison_df["Accuracy"].values
colors = plt.cm.viridis(np.linspace(0, 1, len(models)))
bars = ax.bar(range(len(models)), accuracy, color=colors, alpha=0.7, edgecolor="black")
ax.set_ylabel("Accuracy", fontsize=12, fontweight="bold")
ax.set_title("Accuracy Comparison", fontsize=12, fontweight="bold")
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=45, ha="right", fontsize=9)
ax.set_ylim([0, 1])
ax.grid(axis="y", alpha=0.3)
for i, (bar, val) in enumerate(zip(bars, accuracy)):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.3f}", 
            ha="center", va="bottom", fontsize=9, fontweight="bold")

# F1-Score (Macro) comparison
ax = axes[0, 1]
f1_macro = comparison_df["F1-Score (Macro)"].values
bars = ax.bar(range(len(models)), f1_macro, color=colors, alpha=0.7, edgecolor="black")
ax.set_ylabel("F1-Score (Macro)", fontsize=12, fontweight="bold")
ax.set_title("F1-Score (Macro) Comparison", fontsize=12, fontweight="bold")
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=45, ha="right", fontsize=9)
ax.set_ylim([0, 1])
ax.grid(axis="y", alpha=0.3)
for i, (bar, val) in enumerate(zip(bars, f1_macro)):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.3f}", 
            ha="center", va="bottom", fontsize=9, fontweight="bold")

# F1-Score (Micro) comparison
ax = axes[1, 0]
f1_micro = comparison_df["F1-Score (Micro)"].values
bars = ax.bar(range(len(models)), f1_micro, color=colors, alpha=0.7, edgecolor="black")
ax.set_ylabel("F1-Score (Micro)", fontsize=12, fontweight="bold")
ax.set_title("F1-Score (Micro) Comparison", fontsize=12, fontweight="bold")
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=45, ha="right", fontsize=9)
ax.set_ylim([0, 1])
ax.grid(axis="y", alpha=0.3)
for i, (bar, val) in enumerate(zip(bars, f1_micro)):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.3f}", 
            ha="center", va="bottom", fontsize=9, fontweight="bold")

# Precision vs Recall
ax = axes[1, 1]
precision = comparison_df["Precision (Macro)"].values
recall = comparison_df["Recall (Macro)"].values
x_pos = np.arange(len(models))
width = 0.35
bars1 = ax.bar(x_pos - width/2, precision, width, label="Precision", color="skyblue", alpha=0.8, edgecolor="black")
bars2 = ax.bar(x_pos + width/2, recall, width, label="Recall", color="orange", alpha=0.8, edgecolor="black")
ax.set_ylabel("Score", fontsize=12, fontweight="bold")
ax.set_title("Precision vs Recall (Macro)", fontsize=12, fontweight="bold")
ax.set_xticks(x_pos)
ax.set_xticklabels(models, rotation=45, ha="right", fontsize=9)
ax.set_ylim([0, 1])
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot confusion matrices for each model
num_models = len(validation_results)
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

fig.suptitle("Confusion Matrices - Model Comparison", fontsize=16, fontweight="bold")

for idx, (model_name, metrics) in enumerate(validation_results.items()):
    if idx >= len(axes):
        break
    
    ax = axes[idx]
    cm = metrics["confusion_matrix"].astype(int)
    
    # Normalize confusion matrix for better visualization
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Create heatmap
    im = ax.imshow(cm_normalized, interpolation='nearest', cmap=plt.cm.Blues, aspect='auto')
    
    # Set title and labels
    ax.set_title(f"{model_name}\n(Accuracy: {metrics['accuracy']:.3f})", fontweight="bold")
    ax.set_ylabel("True Label", fontweight="bold")
    ax.set_xlabel("Predicted Label", fontweight="bold")
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Normalized Count", rotation=270, labelpad=15)
    
    # Limit display to top classes if dataset is large
    if cm.shape[0] > 15:
        # Show only first 10 classes for readability
        ax.set_xticks(range(10))
        ax.set_yticks(range(10))
        ax.set_xticklabels(range(10), fontsize=8)
        ax.set_yticklabels(range(10), fontsize=8)
        ax.text(0.5, -0.15, f"(Showing first 10 of {cm.shape[0]} classes)", 
                ha='center', transform=ax.transAxes, fontsize=8, style='italic')
    else:
        ax.set_xticks(range(cm.shape[1]))
        ax.set_yticks(range(cm.shape[0]))
        ax.set_xticklabels(range(cm.shape[1]), fontsize=8)
        ax.set_yticklabels(range(cm.shape[0]), fontsize=8)

# Hide extra subplots if less than 4 models
for idx in range(num_models, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Create detailed metrics summary with radar chart
from math import pi

fig, axes = plt.subplots(1, len(validation_results), figsize=(4*len(validation_results), 5), subplot_kw=dict(projection='polar'))
if len(validation_results) == 1:
    axes = [axes]

fig.suptitle("Comprehensive Model Metrics (Radar Chart)", fontsize=14, fontweight="bold", y=1.02)

# Metrics to include in radar chart
metrics_list = ["Accuracy", "F1-Score (Macro)", "F1-Score (Micro)", "Precision (Macro)", "Recall (Macro)"]
num_metrics = len(metrics_list)
angles = [n / float(num_metrics) * 2 * pi for n in range(num_metrics)]
angles += angles[:1]

colors_radar = plt.cm.Set2(np.linspace(0, 1, len(validation_results)))

for idx, (model_name, metrics) in enumerate(validation_results.items()):
    ax = axes[idx]
    
    # Get metric values
    values = [
        metrics["accuracy"],
        metrics["f1_macro"],
        metrics["f1_micro"],
        metrics["precision_macro"],
        metrics["recall_macro"],
    ]
    values += values[:1]
    
    # Plot
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name, color=colors_radar[idx])
    ax.fill(angles, values, alpha=0.25, color=colors_radar[idx])
    
    # Configure
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics_list, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_title(f"{model_name}", fontsize=10, fontweight="bold", pad=20)

plt.tight_layout()
plt.show()

In [ ]:
# Export results to CSV for future reference
output_dir = Path("/home/mozi30/data/evaluation_results")
output_dir.mkdir(parents=True, exist_ok=True)

# Save comparison dataframe
csv_path = output_dir / "model_comparison.csv"
comparison_df.to_csv(csv_path, index=False)
print(f"✓ Comparison results saved to: {csv_path}")

# Save detailed metrics for each model
for model_name, metrics in validation_results.items():
    # Remove confusion matrix (save separately)
    cm = metrics.pop("confusion_matrix", None)
    metrics_df = pd.DataFrame([metrics])
    
    # Clean model name for filename
    clean_name = model_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    metrics_path = output_dir / f"metrics_{clean_name}.csv"
    metrics_df.to_csv(metrics_path, index=False)
    
    # Save confusion matrix
    if cm is not None:
        cm_path = output_dir / f"confusion_matrix_{clean_name}.npy"
        np.save(cm_path, cm)
    
    print(f"✓ Saved metrics for {model_name}")

print(f"\n✓ All results exported to: {output_dir}")
print(f"  - model_comparison.csv: Summary of all models")
print(f"  - metrics_*.csv: Detailed metrics for each model")
print(f"  - confusion_matrix_*.npy: Confusion matrices for each model")

In [ ]:
# Save `results` summary as CSV
import pandas as pd
import os

OUT_CSV = "/home/mozi30/repos/private/CV_VisionStudio/notebooks/plots/results_summary.csv"

def flatten_dict(d, parent_key='', sep='_'):
    items = {}
    if isinstance(d, dict):
        for k, v in d.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            if isinstance(v, dict):
                items.update(flatten_dict(v, new_key, sep=sep))
            else:
                items[new_key] = v
    return items

rows = []
for key, metrics in results.items():
    try:
        model_name, aug, wt = key
    except Exception:
        # skip malformed keys
        continue
    row = {"model": model_name, "augmentation": aug, "weight_type": wt}

    flat = {}
    if isinstance(metrics, dict):
        flat = flatten_dict(metrics)
    elif isinstance(metrics, (list, tuple)):
        for m in metrics:
            if isinstance(m, dict):
                flat.update(flatten_dict(m))
    elif hasattr(metrics, "__dict__"):
        flat = flatten_dict(vars(metrics))

    for k, v in flat.items():
        # normalize key to safe column name
        col = str(k)
        row[col] = v

    rows.append(row)

if rows:
    df = pd.DataFrame(rows)
    # ensure output dir exists
    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
    df.to_csv(OUT_CSV, index=False)
    print(f"Saved results to {OUT_CSV} ({len(df)} rows)")
else:
    print("No result rows found to save. Check `results` content.")

# mark TODO done
from datetime import datetime
print(f"CSV export completed at {datetime.now().isoformat()}")